In [1]:
from pathlib import Path
import nibabel as nib

base_path = Path(
    "/Users/clairenastaskin/data/Data_processing/2025-03-13/task_movements_run-01_wrist_fingers/LambdaOverTwo/34mm_45mm"
)
file_name = 'sub-Forest001_ses-20250313_task-movements_run-01_acq-68f68c_proc-c5d7b0+6cff9f_pwd.nii.gz'
pwd_path = base_path / file_name

# Load fUSI nifti file and get data into a numpy array
nifti_img = nib.load(pwd_path)
fusi_data = nifti_img.get_fdata()
print(fusi_data.shape)

(151, 69, 73, 243)


In [2]:
import numpy as np
from scipy.ndimage import zoom

# Calculate zoom factors for spatial dimensions (halving each) while keeping time dimension unchanged
spatial_zoom = (0.5, 0.5, 0.5)
zoom_factors = (*spatial_zoom, 1.0)  # Keep temporal dimension at 1.0

# Resample the data using zoom
resampled_data = zoom(fusi_data, zoom_factors, order=1)

print(f"Original shape: {fusi_data.shape}")
print(f"Resampled shape: {resampled_data.shape}")
# Create the output directory if it doesn't exist
resampled_dir = base_path / "resampled_at_lambda"
resampled_dir.mkdir(exist_ok=True)

# Create a new NIfTI image with the resampled data
# Use the original image's affine and header as a base
new_affine = nifti_img.affine.copy()

# Scale the affine matrix to account for the resampling
# Multiply first 3 diagonal elements by 2 since we halved the dimensions
for i in range(3):
    new_affine[i, i] *= 2

resampled_nifti = nib.Nifti1Image(resampled_data, new_affine, nifti_img.header)

# Save the resampled image
output_path = resampled_dir / "fusi_resampled.nii.gz"
nib.save(resampled_nifti, output_path)
print(f"Saved resampled data to: {output_path}")


Original shape: (151, 69, 73, 243)
Resampled shape: (76, 34, 36, 243)
Saved resampled data to: /Users/clairenastaskin/data/Data_processing/2025-03-13/task_movements_run-01_wrist_fingers/LambdaOverTwo/34mm_45mm/resampled_at_lambda/fusi_resampled.nii.gz
